# Time Series Modeling with Satellite Data #

This notebook will explore the modeling and predicting soil moisture data obtained through soil stations and satellite data using ARIMA and neural networks.

https://www.tensorflow.org/tutorials/structured_data/time_series

In [110]:
import os
import datetime

import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, metrics, losses

import warnings
warnings.filterwarnings("ignore")

## Download data ##

In [92]:
folder_addr = "./merged_data/"
merged_sm_list = {}
for i in range(6):
    df = pd.read_csv(folder_addr + 'Station' + str(i + 1) + '_Merged.csv')
    merged_sm_list['Station' + str(i + 1)] = df

In [93]:
merged_sm_list['Station1'].head()

,Date,Sat_SM,Ppt,Tair,RH,Windspeed,Winddirection,Srad,Latitude,Longitude
0,2017-03-18,0.224324,0.013512,15.396964,74.542381,2.214155,150.065714,172.204821,30.3989,-98.6105
1,2017-03-19,0.191104,0.003036,17.210798,70.176905,2.314351,172.232857,204.373869,30.3989,-98.6105
2,2017-03-20,0.210310,0.000000,18.358595,68.807440,2.513542,194.085714,215.361131,30.3989,-98.6105
3,2017-03-21,0.168200,0.000000,19.392440,70.173571,2.653387,195.814286,213.818155,30.3989,-98.6105
4,2017-03-22,0.167357,0.000000,20.166548,70.561905,2.729702,196.057143,221.869821,30.3989,-98.6105


## Feature Engineering ##

In [94]:
day = 24 * 60 * 60
year = 365.2425 * day

for station in merged_sm_list.keys():
    df = merged_sm_list[station]
    
    dates = df.pop('Date')
    timestamp_s = pd.to_datetime(dates).map(pd.Timestamp.timestamp)
    wv = df.pop('Windspeed')
    lat = df.pop('Latitude')
    lon = df.pop('Longitude')
    wd_rad = df.pop('Winddirection') * np.pi / 180
    
    df['Wx'] = wv * np.cos(wd_rad)
    df['Wy'] = wv * np.sin(wd_rad)
    df['Year sin'] = np.sin(timestamp_s * (2 * np.pi / year))
    df['Year cos'] = np.cos(timestamp_s * (2 * np.pi / year))
    df["x_cord"] = np.cos(lat) * np.cos(lon)
    df["y_cord"] = np.sin(lat) * np.cos(lon)
    df["z_cord"] = np.sin(lon)
    merged_sm_list[station] = df

In [95]:
merged_sm_list['Station1'].head()

,Sat_SM,Ppt,Tair,RH,Srad,Wx,Wy,Year sin,Year cos,x_cord,y_cord,z_cord
0,0.224324,0.013512,15.396964,74.542381,172.204821,-1.918783,1.104877,0.968161,0.250330,-0.180165,0.291386,0.939486
1,0.191104,0.003036,17.210798,70.176905,204.373869,-2.293118,0.312779,0.972324,0.233638,-0.180165,0.291386,0.939486
2,0.210310,0.000000,18.358595,68.807440,215.361131,-2.437966,-0.611729,0.976199,0.216878,-0.180165,0.291386,0.939486
3,0.168200,0.000000,19.392440,70.173571,213.818155,-2.552956,-0.723101,0.979785,0.200053,-0.180165,0.291386,0.939486
4,0.167357,0.000000,20.166548,70.561905,221.869821,-2.623207,-0.755025,0.983081,0.183169,-0.180165,0.291386,0.939486


## Split and normalize methods ##

In [96]:
def split(df, target_col='Sat_SM', train_split=0.7, val_split = 0.2):
    n = len(df)
    train_df = df[:int(n*train_split)]
    val_df = df[int(n*train_split):int(n*(train_split + val_split))]
    test_df = df[int(n*(train_split + val_split)):]
    return (train_df, val_df, test_df)

In [97]:
def split_and_normalize(df):
    # Split the data
    train_df, val_df, test_df = split(df)
    
    # Normalize the data
    train_mean = train_df.mean()
    train_std = train_df.std()

    norm_train_df = (train_df - train_mean) / train_std
    norm_val_df = (val_df - train_mean) / train_std
    norm_test_df = (test_df - train_mean) / train_std
    
    return norm_train_df, norm_val_df, norm_test_df

## Data windowing ##

In [98]:
def generate_windows(data_df, input_width=7, shift=7, label_width=1, target_col='Sat_SM'):
    input_data = data_df.drop(target_col, axis=1).values
    all_labels = data_df[target_col].values

    X = []
    y = []
    total_window_size = input_width + shift + label_width
    for i in range(len(input_data) - total_window_size):
        # get window based on input width
        inputs = input_data[i:i+input_width]

        # keep track of labels associated with current window
        label_start_idx = i + input_width + shift
        labels = all_labels[label_start_idx:label_start_idx+label_width]

        X.append(inputs)
        y.append(labels)

    return np.array(X), np.array(y)

In [99]:
def generate_batches(X, y, batch_size=32):
    tf_dataset = tf.data.Dataset.from_tensor_slices((X, y))
    batched_dataset = tf_dataset.repeat().batch(batch_size=batch_size, drop_remainder=True)

    # tf_dataset repeats indefinitely, need to compute number of step updates to complete 1 epoch
    steps_per_epoch = len(X) // batch_size

    return (batched_dataset, steps_per_epoch)

## Make final datasets ##

In [139]:
def make_final_datasets(df):
    train_df, val_df, test_df = split_and_normalize(df)
    
    X_train, y_train = generate_windows(train_df)
    X_val, y_val = generate_windows(val_df)
    X_test, y_test = generate_windows(test_df)

    return {
        'X_train': X_train,
        'y_train': y_train,
        'X_val': X_val,
        'y_val': y_val,
        'X_test': X_test,
        'y_test': y_test
    }

In [140]:
final_data = {}
for station, df in merged_sm_list.items():
    final_data[station] = make_final_datasets(df)

## Models ##

In [142]:
loss_by_epoch = {}
val_performance = {}
performance = {}

station = 'Station1'

X_train = final_data[station]['X_train']
y_train = final_data[station]['y_train']

X_val = final_data[station]['X_val']
y_val = final_data[station]['y_val']

X_test = final_data[station]['X_test']
y_test = final_data[station]['y_test']

In [143]:
def train_model(model, X_train, y_train, X_val, y_val, criterion, optimizer, batch_size=32, nepoch=100):
    train_set, train_steps = generate_batches(X_train, y_train)
    val_set, val_steps = generate_batches(X_val, y_val)

    model.compile(loss=criterion, optimizer=optimizer)
    history = model.fit(train_set,
                        epochs=nepoch,
                        validation_data=val_set,
                        validation_steps=val_steps,
                        steps_per_epoch=train_steps,
                        verbose=1)
    
    loss_by_epoch = history.history
    val_performance = model.evaluate(val_set, steps=val_steps, batch_size=batch_size, verbose=1)
    return loss_by_epoch, val_performance

In [147]:
def test_model(model, X_test, y_test, batch_size=32):
    test_set, test_steps = generate_batches(X_test, y_test)
    performance = model.evaluate(test_set, steps=test_steps, batch_size=batch_size, verbose=0)
    return performance

### LSTM ###

In [145]:
lstm_model = tf.keras.models.Sequential([
    layers.LSTM(128, return_sequences=True),
    layers.LSTM(64, return_sequences=True),
    layers.LSTM(32, return_sequences=False),
    layers.Dense(units=32, activation='relu'),
    layers.Dense(units=1, activation='tanh')
])

In [146]:
criterion = losses.MeanSquaredError()
optimizer = optimizers.Adam(learning_rate=1e-3)

train_loss, val_loss = train_model(lstm_model, X_train, y_train, X_val, y_val, criterion, optimizer)
performance = test_model(lstm_model, X_test, y_test)

loss_by_epoch["LSTM"] = train_loss
val_performance["LSTM"] = val_loss
performance["LSTM"] = performance

Epoch 1/100


2024-04-24 22:23:06.624724: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


29/29 [==============================] - 8s 139ms/step - loss: 0.9369 - val_loss: 1.1010
Epoch 2/100
29/29 [==============================] - 1s 35ms/step - loss: 0.8420 - val_loss: 1.0816
Epoch 3/100
29/29 [==============================] - 1s 29ms/step - loss: 0.7719 - val_loss: 1.1545
Epoch 4/100
29/29 [==============================] - 1s 30ms/step - loss: 0.7546 - val_loss: 1.1221
Epoch 5/100
29/29 [==============================] - 1s 29ms/step - loss: 0.7628 - val_loss: 1.1484
Epoch 6/100
29/29 [==============================] - 1s 32ms/step - loss: 0.7275 - val_loss: 1.1434
Epoch 7/100
29/29 [==============================] - 1s 29ms/step - loss: 0.7202 - val_loss: 1.1405
Epoch 8/100
29/29 [==============================] - 1s 30ms/step - loss: 0.7110 - val_loss: 1.1458
Epoch 9/100
29/29 [==============================] - 1s 29ms/step - loss: 0.7337 - val_loss: 1.0809
Epoch 10/100
29/29 [==============================] - 1s 29ms/step - loss: 0.6979 - val_loss: 1.1041
Epoch 11/1

29/29 [==============================] - 1s 30ms/step - loss: 0.4129 - val_loss: 1.0813
Epoch 83/100
29/29 [==============================] - 1s 29ms/step - loss: 0.4401 - val_loss: 1.1621
Epoch 84/100
29/29 [==============================] - 1s 29ms/step - loss: 0.4341 - val_loss: 0.9754
Epoch 85/100
29/29 [==============================] - 1s 29ms/step - loss: 0.4444 - val_loss: 1.1146
Epoch 86/100
29/29 [==============================] - 1s 29ms/step - loss: 0.4351 - val_loss: 1.0294
Epoch 87/100
29/29 [==============================] - 1s 30ms/step - loss: 0.4474 - val_loss: 1.0952
Epoch 88/100
29/29 [==============================] - 1s 29ms/step - loss: 0.4373 - val_loss: 1.0649
Epoch 89/100
29/29 [==============================] - 1s 29ms/step - loss: 0.4342 - val_loss: 1.0549
Epoch 90/100
29/29 [==============================] - 1s 29ms/step - loss: 0.4287 - val_loss: 1.0626
Epoch 91/100
29/29 [==============================] - 1s 29ms/step - loss: 0.4253 - val_loss: 1.0289
Epo

TypeError: 'float' object does not support item assignment

In [138]:
performance["LSTM"]

2.368565320968628